In [23]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import warnings  # To suppress noisy warnings while keeping the code beginner-friendly
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt  # Plotting
import seaborn as sns  # Nice statistical plots

from sklearn.model_selection import StratifiedKFold  # Robust CV on imbalanced/shifted targets
from sklearn.metrics import roc_auc_score, log_loss  # Key metrics for probabilistic classifiers
from sklearn.preprocessing import StandardScaler  # Scaling numeric features

# We’ll use LightGBM for strong tabular performance and robustness.
# If not installed: pip install lightgbm
from lightgbm import LGBMClassifier 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

%load_ext sql
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/diabetes-health-indicators-dataset/diabetes_dataset.csv
/kaggle/input/playground-series-s5e12/sample_submission.csv
/kaggle/input/playground-series-s5e12/train.csv
/kaggle/input/playground-series-s5e12/test.csv
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [24]:
import csv, sqlite3
import prettytable

prettytable.DEFAULT = 'DEFAULT'

con = sqlite3.connect("my_data3.db")
cur = con.cursor()

In [25]:
%sql sqlite:///my_data3.db

# Load the data

In [26]:
train = pd.read_csv('/kaggle/input/playground-series-s5e12/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e12/test.csv')
org = pd.read_csv('/kaggle/input/diabetes-health-indicators-dataset/diabetes_dataset.csv')

In [27]:
train.drop('id', axis=1, inplace=True)
test.drop('id', axis=1, inplace=True)

In [12]:
org.shape

(100000, 31)

In [28]:
org.to_sql('DIABETESTBL', con, if_exists='replace', index=False, chunksize=200)

100000

### Select all records from <code>DIABETESTBL</code>

In [29]:
%sql select * from DIABETESTBL LIMIT 5;

 * sqlite:///my_data3.db
Done.


age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,family_history_diabetes,hypertension_history,cardiovascular_history,bmi,waist_to_hip_ratio,systolic_bp,diastolic_bp,heart_rate,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,7.9,7.9,0,0,0,30.5,0.89,134,78,68,239,41,160,145,136,236,6.36,8.18,29.6,Type 2,1
48,Female,White,Highschool,Middle,Employed,Former,1,143,6.7,6.5,8.7,0,0,0,23.1,0.8,129,76,67,116,55,50,30,93,150,2.0,5.63,23.0,No Diabetes,0
60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,6.4,10.0,8.1,1,0,0,22.2,0.81,115,73,74,213,66,99,36,118,195,5.07,7.51,44.7,Type 2,1
74,Female,Black,Highschool,Low,Retired,Never,0,49,3.4,6.6,5.2,0,0,0,26.8,0.88,120,93,68,171,50,79,140,139,253,5.28,9.03,38.2,Type 2,1
46,Male,White,Graduate,Middle,Retired,Never,1,109,7.2,7.4,5.0,0,0,0,21.2,0.78,92,67,67,210,52,125,160,137,184,12.74,7.2,23.5,Type 2,1


In [36]:
%sql select MAX(age) - MIN(age) as diff_age from DIABETESTBL;

 * sqlite:///my_data3.db
Done.


diff_age
72


### What is the max physical activity minutes per week?

In [34]:
%sql select MAX(physical_activity_minutes_per_week) from DIABETESTBL;

 * sqlite:///my_data3.db
Done.


MAX(physical_activity_minutes_per_week)
833


### What is the average sleep hours per day?

In [37]:
%sql select AVG(sleep_hours_per_day) from DIABETESTBL;

 * sqlite:///my_data3.db
Done.


AVG(sleep_hours_per_day)
6.997818000000057


In [45]:
%sql select LENGTH(ethnicity) from DIABETESTBL ORDER BY LENGTH(ethnicity) limit 10;

 * sqlite:///my_data3.db
Done.


LENGTH(ethnicity)
5
5
5
5
5
5
5
5
5
5


#### Select five rows where ethnicity is Asian.

In [16]:
%sql select * from DIABETESTBL WHERE ethnicity = 'Asian' LIMIT 10;

 * sqlite:///my_data3.db
Done.


age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,family_history_diabetes,hypertension_history,cardiovascular_history,bmi,waist_to_hip_ratio,systolic_bp,diastolic_bp,heart_rate,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,7.9,7.9,0,0,0,30.5,0.89,134,78,68,239,41,160,145,136,236,6.36,8.18,29.6,Type 2,1
22,Male,Asian,Highschool,Lower-Middle,Employed,Never,3,105,7.0,9.2,1.9,0,0,0,31.4,0.91,113,69,62,188,47,100,131,105,152,16.58,5.88,21.1,Pre-Diabetes,0
53,Female,Asian,Highschool,Lower-Middle,Employed,Former,0,78,7.4,6.8,9.0,0,1,0,23.4,0.85,124,60,65,176,65,74,82,112,158,9.23,6.42,26.6,Pre-Diabetes,0
43,Male,Asian,Highschool,Upper-Middle,Employed,Never,1,175,6.2,7.0,5.1,0,0,0,24.5,0.81,98,90,71,156,48,64,73,106,137,13.54,5.92,21.4,Pre-Diabetes,0
22,Male,Asian,Highschool,Upper-Middle,Employed,Former,2,77,5.6,6.1,9.3,1,0,0,31.3,0.88,112,90,61,182,44,87,143,114,163,5.02,6.66,41.5,Type 2,1
65,Female,Asian,Postgraduate,Middle,Retired,Never,1,56,8.7,6.6,5.1,0,0,0,24.1,0.83,127,90,63,227,63,132,71,106,199,2.0,7.17,29.3,Type 2,1
47,Female,Asian,Graduate,Middle,Retired,Never,1,180,5.4,6.3,5.9,0,1,0,24.1,0.85,124,87,66,182,55,92,92,97,122,9.84,5.44,22.1,No Diabetes,0
49,Female,Asian,No formal,Middle,Unemployed,Never,3,59,7.9,7.5,10.4,0,1,0,24.8,0.82,120,70,73,175,51,100,72,105,156,2.0,6.78,28.1,Type 2,1
74,Female,Asian,No formal,Middle,Employed,Former,1,99,6.7,5.9,2.3,0,0,0,28.6,0.84,148,90,84,243,50,143,131,120,155,10.58,6.97,34.4,Type 2,1
37,Male,Asian,Highschool,Upper-Middle,Employed,Never,2,228,7.1,7.7,7.7,1,0,0,22.4,0.79,90,69,60,162,38,112,223,127,216,7.28,8.19,36.3,Type 2,1


In [ ]:
%sql select * from DIABETESTBL WHERE physical_activity_minutes_per_week 

In [20]:
org['ethnicity'].unique()

array(['Asian', 'White', 'Hispanic', 'Black', 'Other'], dtype=object)

In [21]:
%sql select gender, education_level, income_level, employment_status from DIABETESTBL WHERE ethnicity = 'Black' LIMIT 10;

 * sqlite:///my_data3.db
Done.


gender,education_level,income_level,employment_status
Female,Highschool,Low,Retired
Male,Highschool,Lower-Middle,Employed
Female,Graduate,Lower-Middle,Retired
Female,Highschool,Middle,Unemployed
Female,Graduate,Lower-Middle,Retired
Male,Highschool,Lower-Middle,Employed
Female,Highschool,Middle,Employed
Female,Highschool,Middle,Employed
Male,Highschool,Middle,Employed
Male,Graduate,Lower-Middle,Student


In [ ]:
target = 'diagnosed_diabetes'
cat_cols = test.select_dtypes(exclude='number').columns.to_list()
base = [col for col in train.columns if col not in [target]]
num_cols = [col for col in base if col not in cat_cols]